In [3]:
import tensorflow as tf

I0000 00:00:1790435041.989353   15021 cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.
I0000 00:00:1790435048.890002   15021 cpu_feature_guard.cc:227] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
I0000 00:00:1790435055.941735   15021 cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.


In [32]:
daily_sales_numbers = [21, 22, -108, 31, -1, 32, 34, 31]

In [33]:
# build a simple tensorflow datgaset 
tf_dataset = tf.data.Dataset.from_tensor_slices(daily_sales_numbers)

In [34]:
for sales in tf_dataset.as_numpy_iterator():
    print(sales)

21
22
-108
31
-1
32
34
31


In [35]:
# select specific elements
for sales in tf_dataset.take(3):
    print(sales.numpy())

21
22
-108


In [36]:
# apply transofrmations

tf_dataset = tf_dataset.filter(lambda x: x > 0)

for sales in tf_dataset:
    print(sales.numpy())

21
22
31
32
34
31


In [37]:
# convert this dollar into RWF
tf_dataset = tf_dataset.map(lambda x: x * 1400)

for sales in tf_dataset:
    print(sales.numpy())

29400
30800
43400
44800
47600
43400


In [38]:
# randomly rearrage the element
tf_dataset = tf_dataset.shuffle(3)

for sales in tf_dataset:
    print(sales.numpy())

43400
44800
29400
43400
47600
30800


In [39]:
tf_dataset = tf_dataset.batch(2)

for sales in tf_dataset:
    print(sales.numpy())

[30800 43400]
[47600 29400]
[44800 43400]


In [55]:
# doing all this input pipeline in single line
tf_datasets = tf.data.Dataset.from_tensor_slices(daily_sales_numbers)

for slaes in tf_datasets:
    print(sales.numpy())

[47600 43400 30800]
[47600 43400 30800]
[47600 43400 30800]
[47600 43400 30800]
[47600 43400 30800]
[47600 43400 30800]
[47600 43400 30800]
[47600 43400 30800]


In [56]:
tf_datasets = tf_datasets.filter(lambda y: y > 0).map(lambda z: z * 1400).shuffle(3).batch(3)

for sales in tf_datasets:
    print(sales.numpy())

[29400 43400 44800]
[47600 43400 30800]


In [49]:
# performing input pipeline with images dataset

In [4]:
image_ds = tf.data.Dataset.list_files("image_dataset/*/*", shuffle=False)

E0000 00:00:1790435063.756632   15021 cuda_platform.cc:52] failed call to cuInit: INTERNAL: CUDA error: Failed call to cuInit: UNKNOWN ERROR (303)


In [5]:
for image in image_ds.take(5):
    print(image)

tf.Tensor(b'image_dataset/lionel_messi/02453493f852c0efe8ff951dea5845c4.jpg', shape=(), dtype=string)
tf.Tensor(b'image_dataset/lionel_messi/10_MESSI_JUGADORS-WEB.jpg', shape=(), dtype=string)
tf.Tensor(b'image_dataset/lionel_messi/1211643746.jpg.0.jpg', shape=(), dtype=string)
tf.Tensor(b'image_dataset/lionel_messi/136054219.jpg.0.jpg', shape=(), dtype=string)
tf.Tensor(b'image_dataset/lionel_messi/190703-lionel-messi.jpg', shape=(), dtype=string)


In [6]:
image_ds = image_ds.shuffle(200)

for image in image_ds.take(5):
    print(image)

tf.Tensor(b'image_dataset/lionel_messi/5cb62e40230000c2006db3ac.jpg', shape=(), dtype=string)
tf.Tensor(b'image_dataset/roger_federer/Federer-US-Open-847483.jpg', shape=(), dtype=string)
tf.Tensor(b'image_dataset/lionel_messi/IMG_20200131_151915.jpg', shape=(), dtype=string)
tf.Tensor(b'image_dataset/lionel_messi/MESSI-On-Field-6-4340-8x10.jpg', shape=(), dtype=string)
tf.Tensor(b'image_dataset/lionel_messi/5e4bac9b3b62b77617608663.jfif', shape=(), dtype=string)


In [7]:
class_names = ["Messi", "Roger"]

In [8]:
# splitting dataset into test, and train dataset
image_count = len(image_ds)
image_count

119

In [9]:
train_ds = image_ds.take(int(image_count * 0.8))
test_ds = image_ds.skip(int(image_count * 0.8))

In [10]:
len(train_ds)

95

In [11]:
len(test_ds)

24

In [12]:
# getting labels in our image dataset
sample = 'image_dataset/roger_federer/rogerfederer_perfect_82463094_646872489387563_6528719341074753867_n.jpg'

sample.split("/")[1]

'roger_federer'

In [13]:
def get_labels(file_path):
    import os
    return tf.strings.split(file_path, os.path.sep)[1]

In [21]:
# process the image to get image_numbers and label
def process_image(file_path):
    label = get_labels(file_path)
    img = tf.io.read_file(file_path) # read image from disk
    img = tf.image.decode_jpeg(img) # decode image or convert it into numbers
    img = tf.image.resize(img, [128, 128])
    return img, label

In [14]:
get_labels('image_dataset/roger_federer/rogerfederer_perfect_82463094_646872489387563_6528719341074753867_n.jpg')

<tf.Tensor: shape=(), dtype=string, numpy=b'roger_federer'>

In [16]:
for labels in train_ds.map(get_labels).take(5):
    print(labels)

tf.Tensor(b'lionel_messi', shape=(), dtype=string)
tf.Tensor(b'lionel_messi', shape=(), dtype=string)
tf.Tensor(b'lionel_messi', shape=(), dtype=string)
tf.Tensor(b'lionel_messi', shape=(), dtype=string)
tf.Tensor(b'roger_federer', shape=(), dtype=string)


In [23]:
train_ds = train_ds.map(process_image)
for labels, img in train_ds.take(3):
    print("Labels", labels)
    print("Images", img)

Labels tf.Tensor(
[[[ 55.53125  46.53125  39.53125]
  [ 56.34375  43.34375  34.875  ]
  [ 64.46875  47.46875  37.46875]
  ...
  [121.78125  69.5625  108.9375 ]
  [ 98.       52.53125  92.59375]
  [ 84.59375  46.125    87.28125]]

 [[ 43.3125   33.3125   32.3125 ]
  [ 48.65625  34.65625  31.71875]
  [ 60.71875  42.71875  38.71875]
  ...
  [105.09375  54.59375 113.59375]
  [ 85.40625  42.96875  94.46875]
  [ 65.15625  30.65625  75.15625]]

 [[ 40.5      32.5      29.5    ]
  [ 41.25     30.375    24.4375 ]
  [ 53.       38.875    31.4375 ]
  ...
  [ 95.59375  43.59375 115.59375]
  [ 83.875    38.96875 104.9375 ]
  [ 61.90625  25.9375   85.90625]]

 ...

 [[ 14.       14.       14.0625 ]
  [ 12.125    12.15625  11.     ]
  [ 22.125    17.84375  44.375  ]
  ...
  [ 20.78125  21.28125 106.78125]
  [ 18.125    17.625   107.1875 ]
  [ 20.       19.5     111.5    ]]

 [[ 11.6875   14.65625  16.75   ]
  [ 13.9375   15.0625    9.9375 ]
  [ 22.125    17.5625   35.8125 ]
  ...
  [ 18.9375   18.5  

In [43]:
def scale_image(img, label):
    return img / 255, labels

In [44]:
train_ds = train_ds.map(scale_image)

for img, label in train_ds.take(5):
    print("Label", label)
    print("Image", img.numpy()[0][0])

Label tf.Tensor(
[[[ 63.77478    34.         36.       ]
  [ 64.0741     34.19922    36.19922  ]
  [ 63.8739     33.524536   38.19922  ]
  ...
  [ 10.524536   10.524536   12.524536 ]
  [ 65.01782    65.01782    67.01782  ]
  [ 19.94397    17.94397    18.94397  ]]

 [[ 89.914185   76.672      84.69922  ]
  [ 81.944336   68.45215    76.55408  ]
  [ 38.724243   30.05957    30.868164 ]
  ...
  [ 45.154175   42.361206   40.958862 ]
  [ 22.917236   20.124268   18.721924 ]
  [ 15.402344   13.402344   14.402344 ]]

 [[ 47.549683   34.976562   36.41797  ]
  [ 68.30762    13.12915    19.282715 ]
  [114.01709    56.15149    62.147583 ]
  ...
  [  2.9882812  27.964722   21.968628 ]
  [ 73.6449     80.10852    75.95288  ]
  [ 18.570312   14.5703125  15.5703125]]

 ...

 [[155.38623   217.67139    89.52966  ]
  [121.995605  187.15576    42.35437  ]
  [ 87.19922   149.60779    21.375    ]
  ...
  [ 93.21643   152.81018    49.9292   ]
  [106.56848   167.56848    46.87976  ]
  [ 95.08655   153.8053    